# Banking Intent — Fine-tune Qwen3-8B on Kaggle

**Before running:**
1. Kaggle → Add-ons → **Secrets**: add `HF_TOKEN` (HuggingFace token with write access)
2. Notebook settings → **Accelerator**: GPU T4 x2 or P100
3. Notebook settings → **Internet**: On
4. Fill in `GITHUB_REPO_URL` and `HF_REPO_ID` in the next cell

In [ ]:
# === CONFIG — change these two lines ===
GITHUB_REPO_URL = "https://github.com/nguyenvmthien/banking-intent-unsloth.git"
HF_REPO_ID      = "minhthien/banking-intent-unsloth"
# ========================================

import os
WORKING_DIR = f"/kaggle/working/{GITHUB_REPO_URL.rstrip('/').split('/')[-1].removesuffix('.git')}"
print(f"Repo will be cloned to: {WORKING_DIR}")

## 1. Environment setup

In [ ]:
import subprocess, sys

def run(cmd, capture=True):
    result = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if result.returncode != 0:
        print(result.stderr[-3000:] if capture else "")
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout if capture else ""

# Unsloth must be installed before transformers to avoid version conflicts
run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
run('pip install -q trl peft datasets langsmith huggingface_hub pandas scikit-learn tqdm pyyaml')
print("Dependencies installed.")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
print("HF_TOKEN loaded.")

## 2. Clone repo and prepare data

In [ ]:
run(f'git clone {GITHUB_REPO_URL} {WORKING_DIR}')
os.chdir(f"{WORKING_DIR}/scripts")
print(f"Repo cloned to {WORKING_DIR}")

In [ ]:
run('python preprocess_data.py')
print("Data ready.")

## 3. Configure HF repo for checkpoint push

In [ ]:
import yaml

config_path = f"{WORKING_DIR}/configs/train.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

config['hub_model_id'] = HF_REPO_ID

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"hub_model_id set to: {HF_REPO_ID}")

## 4. Train

If this cell is re-run after a session restart, it will **automatically resume** from the latest local checkpoint (if any), or pull from Hub if disk was wiped.

In [ ]:
# If Kaggle disk was wiped, restore latest checkpoint from Hub before training
import glob
from huggingface_hub import snapshot_download

output_dir = f"{WORKING_DIR}/outputs/checkpoint"
os.makedirs(output_dir, exist_ok=True)

has_local_checkpoint = bool(glob.glob(os.path.join(output_dir, "checkpoint-*")))

if not has_local_checkpoint:
    try:
        print(f"No local checkpoint. Trying to restore from Hub: {HF_REPO_ID}")
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=output_dir,
            token=os.environ["HF_TOKEN"],
        )
        print("Restored from Hub.")
    except Exception as e:
        print(f"Hub restore skipped ({e}) — starting fresh.")
else:
    latest = sorted(glob.glob(os.path.join(output_dir, "checkpoint-*")))[-1]
    print(f"Local checkpoint found: {latest}")

In [ ]:
os.chdir(f"{WORKING_DIR}/scripts")

# Run training with live output (no capture)
train_result = subprocess.run([sys.executable, "train.py"], env=os.environ)
if train_result.returncode != 0:
    raise RuntimeError("Training failed — check logs above.")
print("Training complete.")

## 5. Quick sanity check — predict one sample

In [ ]:
import sys
sys.path.insert(0, f"{WORKING_DIR}/scripts")

from inference import IntentClassification

clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="finetuned")
test_input = "I lost my credit card, how do I order a replacement?"
result = clf.predict(test_input)
print(result)

## 6. Evaluate on full test set (3,080 samples)

Quick sanity check on 20 samples first, then full evaluation on all modes.

In [ ]:
import sys
import yaml
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

sys.path.insert(0, f"{WORKING_DIR}/scripts")
from inference import IntentClassification

with open(f"{WORKING_DIR}/configs/inference.yaml") as f:
    config = yaml.safe_load(f)
batch_size = config.get("batch_size", 8)

test_df = pd.read_csv(f"{WORKING_DIR}/sample_data/test.csv")


def sample_df(n=20, random_state=42):
    """Return n stratified samples from test_df (one per intent, then random sample)."""
    return (
        test_df.groupby("intent_name", group_keys=False)
        .apply(lambda g: g.sample(min(1, len(g)), random_state=random_state), include_groups=False)
        .sample(min(n, test_df["intent_name"].nunique()), random_state=random_state)
        .reset_index(drop=True)
    )


def run_eval(clf, df, label, print_per_sample=True):
    """Run batched evaluation and print results. Returns accuracy."""
    texts = df["text"].tolist()
    y_true = df["intent_name"].tolist()
    y_pred = []
    for start in tqdm(range(0, len(texts), batch_size), desc=label):
        results = clf.predict_batch(texts[start:start + batch_size])
        for result, true_label in zip(results, y_true[start:start + batch_size]):
            y_pred.append(result["label"])
            if print_per_sample:
                status = "OK" if result["label"] == true_label else "MISS"
                print(f"  [{status}] true={true_label}  pred={result['label']}", flush=True)
    acc = accuracy_score(y_true, y_pred)
    print(f"\n>>> {label} accuracy: {acc:.4f} ({int(acc*len(df))}/{len(df)})\n", flush=True)
    print(classification_report(y_true, y_pred, digits=4), flush=True)
    return acc


# === Quick check — change n to any number you want ===
N_SAMPLES = 20

print("=" * 60)
print(f"QUICK CHECK ({N_SAMPLES} samples)")
print("=" * 60)
quick_df = sample_df(n=N_SAMPLES)
print(f"Sampled {len(quick_df)} rows across {quick_df['intent_name'].nunique()} intents\n")

quick_results = {}
for mode in ("zero_shot", "finetuned"):
    clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode=mode)
    quick_results[mode] = run_eval(clf, quick_df, mode)

print("=== QUICK CHECK SUMMARY ===")
for mode, acc in quick_results.items():
    print(f"  {mode:<12} {acc:.2f}")

# === Full evaluation ===
print("\n" + "=" * 60)
print(f"FULL EVALUATION ({len(test_df)} samples)")
print("=" * 60)

full_results = {}
for mode in ("zero_shot", "finetuned"):
    clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode=mode)
    full_results[mode] = run_eval(clf, test_df, mode, print_per_sample=False)

print("=== FULL EVALUATION SUMMARY ===")
for mode, acc in full_results.items():
    print(f"  {mode:<12} {acc:.4f}  ({acc*100:.2f}%)")